
# 🔥 FIXED Content-Based Notebook (Before vs After)

This notebook:
1. Shows WRONG input (numbers as text)
2. Fixes it (convert numeric genres → words)
3. Shows BEFORE vs AFTER
4. Applies TF-IDF & BoW correctly
5. Saves cleaned dataset


In [1]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from pathlib import Path


from pathlib import Path


import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA = PROJECT_ROOT / 'data' / 'processed'
courses = pd.read_csv(DATA / 'final_courses.csv')



courses.head()








,COURSE_ID,TITLE,Database,Python,CloudComputing,DataAnalysis,Containers,MachineLearning,ComputerVision,DataScience,BigData,Chatbot,R,BackendDev,FrontendDev,Blockchain,num_genres
0,ML0201EN,robots are coming build iot apps with watson ...,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2
1,ML0122EN,accelerating deep learning with gpu,0,1,0,0,0,1,0,1,0,0,0,0,0,0,3
2,GPXX0ZG0EN,consuming restful services using the reactive ...,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2
3,RP0105EN,analyzing big data in r using apache spark,1,0,0,1,0,0,0,0,1,0,1,0,0,0,4
4,GPXX0Z2PEN,containerizing packaging and running a sprin...,0,0,0,0,1,0,0,0,0,0,0,1,0,0,2


## ❌ BEFORE (Wrong Input with Numbers)

In [2]:

text_before = courses.fillna('').astype(str).agg(' '.join, axis=1)

print("Sample BEFORE:\n")
for i in range(5):
    print(text_before.iloc[i][:200])
    print("-"*50)


Sample BEFORE:

ML0201EN robots are coming  build iot apps with watson  swift  and node red 0 0 0 0 0 0 0 0 0 0 0 1 1 0 2
--------------------------------------------------
ML0122EN accelerating deep learning with gpu 0 1 0 0 0 1 0 1 0 0 0 0 0 0 3
--------------------------------------------------
GPXX0ZG0EN consuming restful services using the reactive jax rs client 0 0 0 0 0 0 0 0 0 0 0 1 1 0 2
--------------------------------------------------
RP0105EN analyzing big data in r using apache spark 1 0 0 1 0 0 0 0 1 0 1 0 0 0 4
--------------------------------------------------
GPXX0Z2PEN containerizing  packaging  and running a spring boot application 0 0 0 0 1 0 0 0 0 0 0 1 0 0 2
--------------------------------------------------


## 🔍 Detect Genre Columns

In [3]:

# assume numeric columns = genre indicators
genre_cols = courses.select_dtypes(include=['int64','float64']).columns.tolist()

print("Detected genre columns:", genre_cols[:10])


Detected genre columns: ['Database', 'Python', 'CloudComputing', 'DataAnalysis', 'Containers', 'MachineLearning', 'ComputerVision', 'DataScience', 'BigData', 'Chatbot']


## ✅ AFTER (Convert Numbers → Words)

In [4]:



genre_cols = [col for col in courses.columns if col not in ['COURSE_ID', 'TITLE', 'num_genres']]

def build_text(row):
    genres = [col for col in genre_cols if row[col] > 0]
    return " ".join(genres)

text_after = courses.apply(build_text, axis=1)

print("Sample AFTER:\n")
for i in range(5):
    print(text_after.iloc[i][:200])
    print("-"*50)


Sample AFTER:

BackendDev FrontendDev
--------------------------------------------------
Python MachineLearning DataScience
--------------------------------------------------
BackendDev FrontendDev
--------------------------------------------------
Database DataAnalysis BigData R
--------------------------------------------------
Containers BackendDev
--------------------------------------------------


## 💾 Save Cleaned Dataset

In [5]:

courses['clean_text'] = text_after

OUTPUT = Path(DATA/ "cleaned_courses.csv")
courses.to_csv(OUTPUT, index=False)

print("Saved cleaned dataset to:", OUTPUT)


Saved cleaned dataset to: /Users/macbook/Downloads/recommender_full_project/data/processed/cleaned_courses.csv


## 🔥 TF-IDF

In [6]:

tfidf = TfidfVectorizer(max_features=100)
tfidf_matrix = tfidf.fit_transform(text_after)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

tfidf_df.head()


,backenddev,bigdata,blockchain,chatbot,cloudcomputing,computervision,containers,dataanalysis,database,datascience,frontenddev,machinelearning,python
0,0.534333,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.845274,0.000000,0.00000
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.591576,0.000000,0.478735,0.64873
2,0.534333,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.845274,0.000000,0.00000
3,0.000000,0.639237,0.0,0.0,0.0,0.0,0.000000,0.550677,0.536778,0.000000,0.000000,0.000000,0.00000
4,0.523734,0.000000,0.0,0.0,0.0,0.0,0.851882,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000


## 🔹 BoW

In [7]:

bow = CountVectorizer(max_features=100)
bow_matrix = bow.fit_transform(text_after)

bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out())

bow_df.head()


,backenddev,bigdata,blockchain,chatbot,cloudcomputing,computervision,containers,dataanalysis,database,datascience,frontenddev,machinelearning,python
0,1,0,0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,1,0,1,1
2,1,0,0,0,0,0,0,0,0,0,1,0,0
3,0,1,0,0,0,0,0,1,1,0,0,0,0
4,1,0,0,0,0,0,1,0,0,0,0,0,0


## 🔥 Comparison

In [8]:

tfidf_density = np.count_nonzero(tfidf_df.values) / tfidf_df.size
bow_density = np.count_nonzero(bow_df.values) / bow_df.size

print("TF-IDF density:", tfidf_density)
print("BoW density:", bow_density)


TF-IDF density: 0.11425707842645953
BoW density: 0.11425707842645953
